# 07 Multivariate Analysis

## Objective

Objective:
Perform multivariate analysis to understand how multiple features interact with each other and with the target variable `Churn`.

This step helps in:
- Identifying feature combinations that are strongly associated with churn
- Detecting redundancy and multicollinearity among predictors
- Finding interaction patterns that may support feature engineering
- Translating bivariate findings into modeling-oriented insights


## Imports


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

project_root = Path.cwd()
if not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import CATEGORICAL_COLUMNS, NUMERICAL_COLUMNS, TARGET_COLUMN
from src.data.data_loader import load_cleaned_data

sns.set_theme(style="whitegrid")


## Report Paths

Store multivariate analysis outputs in dedicated report folders so figures and tables stay organized.


In [ ]:
REPORTS_DIR = project_root / "reports"
MULTIVARIATE_FIGURES_DIR = REPORTS_DIR / "figures" / "multivariate"
MULTIVARIATE_TABLES_DIR = REPORTS_DIR / "tables" / "multivariate"
NUMERICAL_RELATIONSHIPS_FIGURES_DIR = MULTIVARIATE_FIGURES_DIR / "numerical_relationships"
NUMERICAL_RELATIONSHIPS_TABLES_DIR = MULTIVARIATE_TABLES_DIR / "numerical_relationships"
CATEGORICAL_INTERACTIONS_FIGURES_DIR = MULTIVARIATE_FIGURES_DIR / "categorical_interactions"
CATEGORICAL_INTERACTIONS_TABLES_DIR = MULTIVARIATE_TABLES_DIR / "categorical_interactions"
MIXED_INTERACTIONS_FIGURES_DIR = MULTIVARIATE_FIGURES_DIR / "mixed_interactions"
MIXED_INTERACTIONS_TABLES_DIR = MULTIVARIATE_TABLES_DIR / "mixed_interactions"

for path in [
    NUMERICAL_RELATIONSHIPS_FIGURES_DIR,
    NUMERICAL_RELATIONSHIPS_TABLES_DIR,
    CATEGORICAL_INTERACTIONS_FIGURES_DIR,
    CATEGORICAL_INTERACTIONS_TABLES_DIR,
    MIXED_INTERACTIONS_FIGURES_DIR,
    MIXED_INTERACTIONS_TABLES_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

{
    "multivariate_figures": MULTIVARIATE_FIGURES_DIR,
    "multivariate_tables": MULTIVARIATE_TABLES_DIR,
}


## Load Data and Separate Feature Types

Load the cleaned dataset and keep the same target, numerical, and categorical feature groups used in earlier notebooks.


In [ ]:
df = load_cleaned_data()

target_column = TARGET_COLUMN
numerical_columns = [column for column in NUMERICAL_COLUMNS if column in df.columns]
categorical_columns = [column for column in CATEGORICAL_COLUMNS if column in df.columns]

feature_groups = {
    "target_column": target_column,
    "numerical_columns": numerical_columns,
    "categorical_columns": categorical_columns,
}

feature_groups


## Correlation Analysis for Numerical Features

Start with the numerical features to see which variables move together, where redundancy may exist, and whether any pair is strong enough to matter later in feature engineering.


In [ ]:
numerical_correlation = df[numerical_columns].corr().round(3)
numerical_correlation.to_csv(
    NUMERICAL_RELATIONSHIPS_TABLES_DIR / "numerical_correlation_matrix.csv"
)

display(numerical_correlation)

plt.figure(figsize=(8, 6))
sns.heatmap(
    numerical_correlation,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    square=True,
)
plt.title("Numerical Feature Correlation Heatmap")
plt.tight_layout()
plt.savefig(
    NUMERICAL_RELATIONSHIPS_FIGURES_DIR / "numerical_correlation_heatmap.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()


The correlation matrix shows whether the numerical features are carrying distinct information or largely repeating the same customer behavior. In this churn dataset, `TotalCharges` is expected to move closely with `tenure`, because longer-lived customers accumulate more billing history over time. This is useful later when deciding whether to keep both variables, transform them, or treat them as part of the same lifecycle story.


## Scatter Plots for Numerical Relationships Colored by Target

Use scatter plots to see whether churn classes separate more clearly when numerical features are viewed together instead of one at a time.


In [ ]:
numerical_pairs = [
    pair for pair in [
        ("tenure", "MonthlyCharges"),
        ("tenure", "TotalCharges"),
        ("MonthlyCharges", "TotalCharges"),
    ]
    if pair[0] in df.columns and pair[1] in df.columns
]

if numerical_pairs:
    n_cols = 2
    n_rows = (len(numerical_pairs) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5 * n_rows))
    axes = np.atleast_1d(axes).flatten()

    for index, (x_column, y_column) in enumerate(numerical_pairs):
        ax = axes[index]
        sns.scatterplot(
            data=df,
            x=x_column,
            y=y_column,
            hue=target_column,
            alpha=0.6,
            palette="Set2",
            ax=ax,
        )
        ax.set_title(f"{y_column} vs {x_column} by {target_column}")

    for index in range(len(numerical_pairs), len(axes)):
        fig.delaxes(axes[index])

    plt.tight_layout()
    plt.savefig(
        NUMERICAL_RELATIONSHIPS_FIGURES_DIR / "numerical_scatter_relationships.png",
        dpi=150,
        bbox_inches="tight",
    )
    plt.show()
else:
    print("Not enough numerical pairs available for scatter plots.")


In [ ]:
if len(numerical_columns) >= 2:
    pairplot_columns = numerical_columns + [target_column]
    pairplot_grid = sns.pairplot(
        df[pairplot_columns],
        hue=target_column,
        diag_kind="hist",
        corner=True,
        plot_kws={"alpha": 0.45, "s": 28},
        palette="Set2",
    )
    pairplot_grid.fig.suptitle("Pairwise Numerical Relationships by Churn", y=1.02)
    pairplot_grid.fig.savefig(
        NUMERICAL_RELATIONSHIPS_FIGURES_DIR / "numerical_pairplot.png",
        dpi=150,
        bbox_inches="tight",
    )
    plt.show()
else:
    print("Not enough numerical columns available for a pairplot.")


These scatter plots help answer whether churn is better explained by a combination of variables than by any single feature alone. The most important relationship to check is whether churned customers cluster at low `tenure` and low `TotalCharges`, because that would reinforce the idea that early-lifecycle churn is the dominant business pattern. If separation remains weak even in two-dimensional views, the model may need interactions, non-linear transformations, or tree-based methods to capture churn behavior more effectively.


## Categorical Interaction Analysis

Important churn signals often emerge from combinations of service, contract, and payment choices rather than from a single category in isolation. This section compares selected categorical pairs using churn-rate tables and heatmaps.


In [ ]:
categorical_pairs = [
    pair for pair in [
        ("Contract", "PaymentMethod"),
        ("InternetService", "TechSupport"),
        ("InternetService", "OnlineSecurity"),
    ]
    if pair[0] in df.columns and pair[1] in df.columns
]

if categorical_pairs:
    n_cols = 2
    n_rows = (len(categorical_pairs) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5 * n_rows))
    axes = np.atleast_1d(axes).flatten()

    for index, (left_column, right_column) in enumerate(categorical_pairs):
        churn_rate_table = (
            df.groupby([left_column, right_column])[target_column]
            .apply(lambda values: values.eq("Yes").mean() * 100)
            .unstack()
            .round(2)
        )
        churn_rate_table.to_csv(
            CATEGORICAL_INTERACTIONS_TABLES_DIR
            / f"{left_column.lower()}_vs_{right_column.lower()}_churn_rate.csv"
        )

        print(f"\n{left_column} vs {right_column} churn rate (%)")
        display(churn_rate_table)

        ax = axes[index]
        sns.heatmap(
            churn_rate_table,
            annot=True,
            fmt=".1f",
            cmap="YlOrRd",
            cbar=False,
            ax=ax,
        )
        ax.set_title(f"{left_column} vs {right_column} Churn Rate")
        ax.set_xlabel(right_column)
        ax.set_ylabel(left_column)

    for index in range(len(categorical_pairs), len(axes)):
        fig.delaxes(axes[index])

    plt.tight_layout()
    plt.savefig(
        CATEGORICAL_INTERACTIONS_FIGURES_DIR / "categorical_pair_heatmaps.png",
        dpi=150,
        bbox_inches="tight",
    )
    plt.show()
else:
    print("Selected categorical interaction pairs are not available in the dataset.")


These categorical interaction tables show whether high-risk churn categories become even more pronounced when combined. For example, month-to-month contracts may look risky on their own, but the risk can become sharper when paired with a payment method such as `Electronic check`, or when internet-service customers also lack support-related add-ons. This is the kind of pattern that often motivates interaction features or segmented retention strategies.


## Numerical and Categorical Interaction Analysis

Now compare key numerical features within important categorical groups while still splitting by churn. This helps reveal whether the same churn pattern behaves differently across contracts, services, or payment segments.


In [ ]:
mixed_pairs = [
    pair for pair in [
        ("tenure", "Contract"),
        ("MonthlyCharges", "InternetService"),
        ("tenure", "PaymentMethod"),
    ]
    if pair[0] in df.columns and pair[1] in df.columns
]

if mixed_pairs:
    fig, axes = plt.subplots(len(mixed_pairs), 1, figsize=(16, 5 * len(mixed_pairs)))
    axes = np.atleast_1d(axes)

    for index, (numeric_column, category_column) in enumerate(mixed_pairs):
        ax = axes[index]
        sns.boxplot(
            data=df,
            x=category_column,
            y=numeric_column,
            hue=target_column,
            palette="Set2",
            ax=ax,
        )
        ax.set_title(f"{numeric_column} by {category_column} and {target_column}")
        ax.tick_params(axis="x", rotation=20)

    plt.tight_layout()
    plt.savefig(
        MIXED_INTERACTIONS_FIGURES_DIR / "numerical_categorical_boxplots.png",
        dpi=150,
        bbox_inches="tight",
    )
    plt.show()
else:
    print("Selected numerical and categorical interaction pairs are not available in the dataset.")


These mixed plots test whether important numerical churn patterns are stable across customer segments. If low `tenure` remains a strong churn signal inside `Contract` groups, or if `MonthlyCharges` behaves differently across internet-service types, that is a sign that the model may benefit from explicit interaction terms or segmented business actions instead of a one-size-fits-all interpretation.


## Multicollinearity Check

Before modeling, check whether some numerical predictors are too closely related. Strong collinearity can make coefficient-based models harder to interpret and may inflate variance estimates.


In [ ]:
correlation_pairs = []

for left_index, left_column in enumerate(numerical_columns):
    for right_column in numerical_columns[left_index + 1 :]:
        correlation_pairs.append(
            {
                "feature_1": left_column,
                "feature_2": right_column,
                "correlation": round(float(df[left_column].corr(df[right_column])), 4),
                "abs_correlation": round(abs(float(df[left_column].corr(df[right_column]))), 4),
            }
        )

correlation_pairs_df = pd.DataFrame(correlation_pairs).sort_values(
    "abs_correlation", ascending=False
)
correlation_pairs_df.to_csv(
    NUMERICAL_RELATIONSHIPS_TABLES_DIR / "numerical_correlation_pairs.csv",
    index=False,
)

display(correlation_pairs_df)

try:
    from statsmodels.stats.outliers_influence import variance_inflation_factor

    vif_input = df[numerical_columns].dropna().copy()
    vif_rows = []
    for index, column in enumerate(vif_input.columns):
        vif_rows.append(
            {
                "feature": column,
                "vif": round(float(variance_inflation_factor(vif_input.values, index)), 4),
            }
        )

    vif_df = pd.DataFrame(vif_rows).sort_values("vif", ascending=False)
    vif_df.to_csv(
        NUMERICAL_RELATIONSHIPS_TABLES_DIR / "numerical_vif.csv",
        index=False,
    )
    display(vif_df)
except ImportError:
    print("statsmodels is not installed, so VIF was skipped. Correlation review is still available above.")


The multicollinearity check shows whether any numerical variables are close enough that they may compete with each other in linear models. If `tenure` and `TotalCharges` show a strong correlation or high VIF, that does not mean one must be dropped immediately, but it does mean the modeler should think carefully about whether both belong in the same form, or whether one should be transformed, regularized, or replaced with a more interpretable derived feature.


## Interaction-Based Business Insights

- If churn remains concentrated among low-tenure customers across multiple contract or payment segments, early retention interventions should remain the highest operational priority.
- If the riskiest churn rates appear in combinations such as `Month-to-month` plus `Electronic check`, or `Fiber optic` plus weak support adoption, those combinations become strong candidates for targeted campaigns.
- If some numerical relationships remain stable across categories, the model may capture them well without heavy segmentation. If they change sharply by category, interaction features become more important.
- Multivariate analysis is especially valuable when the business needs not just a ranked feature list, but a clearer answer to which customer profiles are most vulnerable to churn.


## Modeling Implications

- Use the correlation and VIF checks to decide whether `tenure` and `TotalCharges` should enter the model together, be regularized, or be represented through engineered lifecycle features.
- Preserve strong categorical drivers such as `Contract`, `InternetService`, `TechSupport`, `OnlineSecurity`, and `PaymentMethod`, because their interactions may capture churn risk better than isolated one-hot variables alone.
- Consider testing interaction terms such as `tenure x MonthlyCharges`, `Contract x PaymentMethod`, and `InternetService x TechSupport` if the multivariate charts reveal concentrated high-risk regions.
- Tree-based models may naturally capture these joint patterns, while linear models may need explicit interaction features and careful encoding to represent them well.


## Final Conclusion

By the end of this notebook, the goal is to identify which numerical features move together, which categorical combinations create especially high-risk churn profiles, whether any predictors look redundant, and which feature interactions are promising enough to carry into modeling. In this project, the key multivariate question is whether the churn story remains centered on short-tenure customers with weaker commitment and lower support adoption even after multiple features are considered jointly.
